# Resume cpsam fine-tune on DIC

Continues training from a partial `cpsam_dic` checkpoint already on Drive.
Use this notebook when an earlier run got cut short by a Colab session
timeout — flow fields are cached on Drive, so the precompute step is
skipped and training picks up where it left off.

**Prerequisites**
- Earlier run produced `MyDrive/cellscope_training/models/cpsam_dic`
- Training data still at `MyDrive/cellscope_training/dic_splits_v3/train/`
- GPU runtime (T4 free works; A100 faster)

**Output**: overwrites `MyDrive/cellscope_training/models/cpsam_dic`
with the further-trained weights. Download and place at
`cellscope/data/models/cpsam_dic` to use in the GUI.

In [ ]:
# Step 1: Install cellpose 4.x (must match the version that wrote the checkpoint)
!pip install 'cellpose>=4.1.1' tifffile -q

In [ ]:
# Step 1b (recommended): keep the Colab session from idle-disconnecting.
from IPython.display import display, Javascript
display(Javascript('''
function ConnectButton() {
    const b = document.querySelector("colab-connect-button");
    if (b) b.click();
}
setInterval(ConnectButton, 60000);
'''))
print('Keep-alive installed.')

In [ ]:
# Step 2: Mount Google Drive and locate the checkpoint
# cellpose's train_seg saves to <save_path>/models/<name>, so a save_path
# of MyDrive/cellscope_training/models writes to
# MyDrive/cellscope_training/models/models/cpsam_dic (doubled `/models/`).
# We probe both layouts so this works either way.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/cellscope_training'
TRAIN_DIR = f'{DRIVE_PATH}/dic_splits_v3/train'

candidates = [
    f'{DRIVE_PATH}/models/models/cpsam_dic',   # train_seg's actual layout
    f'{DRIVE_PATH}/models/cpsam_dic',          # idealised layout
]
CHECKPOINT = next((p for p in candidates if os.path.exists(p)), None)

if CHECKPOINT is None:
    print('Searched:')
    for p in candidates:
        print(f'  - {p}')
    raise SystemExit(
        'No checkpoint found. Run train_cpsam_dic_colab.ipynb first, or '
        'edit the candidates list to add the actual path on your Drive.')

if not os.path.isdir(TRAIN_DIR):
    raise SystemExit(f'No training data at {TRAIN_DIR}')

# save_path for train_seg must be the PARENT of the `models/` dir so the
# resumed weights overwrite the same file. So strip `/models/<name>` once.
SAVE_PATH = os.path.dirname(os.path.dirname(CHECKPOINT))

size_mb = os.path.getsize(CHECKPOINT) / 1024 / 1024
print(f'Checkpoint:  {CHECKPOINT}  ({size_mb:.0f} MB)')
print(f'Save path:   {SAVE_PATH}')
print(f'Training:    {TRAIN_DIR}')


In [ ]:
# Step 3: Collect file paths (same logic as the v1 training notebook)
# Subsample fits Colab T4 free (~13 GB RAM) — flow tensors held in RAM
# scale linearly with pair count.
import glob, tifffile

img_files = sorted(glob.glob(f'{TRAIN_DIR}/*_img.tif'))
train_files, train_labels_files = [], []
for f in img_files:
    mf = f.replace('_img.tif', '_masks.tif')
    if os.path.exists(mf):
        train_files.append(f)
        train_labels_files.append(mf)

# IMPORTANT: keep this the same as the original run if you want to reuse
# the cached flow files on Drive. The same SUBSAMPLE_TO + seed = same
# subset = same flow cache hits. Mismatch → cellpose recomputes flows.
SUBSAMPLE_TO = 1000     # match original run; set None for High-RAM
if SUBSAMPLE_TO and len(train_files) > SUBSAMPLE_TO:
    import random; random.seed(0)
    idx = sorted(random.sample(range(len(train_files)), SUBSAMPLE_TO))
    train_files = [train_files[i] for i in idx]
    train_labels_files = [train_labels_files[i] for i in idx]

# Spot-check + count cached flows so we know the precompute will be fast.
n_flows_cached = sum(
    1 for mf in train_labels_files
    if os.path.exists(mf.replace('_masks.tif', '_flows.tif')))
print(f'Pairs: {len(train_files)}')
print(f'Flow cache hits: {n_flows_cached}/{len(train_files)} '
      f'({100*n_flows_cached/max(len(train_files),1):.0f}%)')
if n_flows_cached < len(train_files) * 0.9:
    print('NOTE: many flows are uncached — first epoch precompute will be slow.')

In [ ]:
# Step 4: Resume training from the existing checkpoint
# Differences vs the v1 notebook:
#  • models.CellposeModel(pretrained_model=CHECKPOINT) loads the
#    partial weights instead of the default cpsam.
#  • N_EPOCHS counts ADDITIONAL epochs to run, not total.
import time, gc, threading
import cellpose
from cellpose import models, train

print(f'cellpose version: {cellpose.version}')
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass

# Resume from the partial weights — KEY DIFFERENCE FROM V1 NOTEBOOK
model = models.CellposeModel(gpu=True, pretrained_model=CHECKPOINT)

N_EPOCHS = 14         # number of ADDITIONAL epochs to run
LR = 1e-5             # same LR as original — keep consistent
BATCH_SIZE = 1        # T4 can't fit cpsam at >1
SAVE_EVERY = 1        # checkpoint after each epoch

OUT_DIR = f'{DRIVE_PATH}/models'

print(f'Resuming from {CHECKPOINT}')
print(f'Adding {N_EPOCHS} epochs at lr={LR}, batch={BATCH_SIZE}, '
      f'save_every={SAVE_EVERY}')
print(f'Output (overwritten each epoch): {OUT_DIR}/cpsam_dic')

# Heartbeat — prints status every 60 s.
def _heartbeat(stop):
    import psutil
    t0 = time.time()
    while not stop.is_set():
        mins = (time.time() - t0) / 60
        ram = psutil.virtual_memory()
        msg = (f'[heartbeat] +{mins:5.1f} min   '
               f'RAM {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB')
        try:
            import torch
            if torch.cuda.is_available():
                msg += f'   GPU {torch.cuda.memory_allocated()/1e9:.1f} GB'
        except Exception:
            pass
        print(msg, flush=True)
        for _ in range(60):
            if stop.is_set():
                return
            time.sleep(1)

stop = threading.Event()
hb = threading.Thread(target=_heartbeat, args=(stop,), daemon=True)
hb.start()

t0 = time.time()
try:
    new_path, train_losses, test_losses = train.train_seg(
        model.net,
        train_files=train_files,
        train_labels_files=train_labels_files,
        save_path=OUT_DIR,
        n_epochs=N_EPOCHS,
        learning_rate=LR,
        batch_size=BATCH_SIZE,
        save_every=SAVE_EVERY,
        model_name='cpsam_dic',
        min_train_masks=1,
    )
finally:
    stop.set()
    hb.join(timeout=2)

print(f'\nDone in {(time.time()-t0)/60:.1f} minutes')
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Model saved: {new_path}')

In [ ]:
# Step 5: Quick validation against the held-out val set (if present)
import numpy as np

VAL_DIR = f'{DRIVE_PATH}/dic_splits_v3/val'
if os.path.isdir(VAL_DIR):
    val_imgs = sorted(glob.glob(f'{VAL_DIR}/*_img.tif'))[:20]
    trained = models.CellposeModel(gpu=True, pretrained_model=new_path)
    ious = []
    for f in val_imgs:
        img = tifffile.imread(f)
        gt = tifffile.imread(f.replace('_img.tif', '_masks.tif')) > 0
        pred = trained.eval(img)[0] > 0
        inter = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        ious.append(inter / union if union > 0 else 0)
    print(f'Validation IoU after resume: {np.mean(ious):.3f} '
          f'({len(val_imgs)} frames)')
else:
    print('No val/ directory — skipping validation')

In [ ]:
# Step 6: Download the model (or copy from Drive locally)
from google.colab import files
try:
    files.download(new_path)
    print('Download complete.')
except Exception as e:
    print(f'Download failed ({e}). The model is still at:')
    print(f'  {new_path}')
    print('  Open Drive in your browser and download from there.')
print('\nPlace the model at:')
print('  cellscope/data/models/cpsam_dic')

## Crashed mid-train again? Just rerun this notebook.

Each time you run cells 1–4, the loop picks up the latest weights from
`MyDrive/cellscope_training/models/cpsam_dic` (which is overwritten
each epoch via `save_every=1`). Flow caches on Drive accumulate, so
repeated re-runs become progressively faster to start.

Adjust `N_EPOCHS` per session based on how much time you have.
Rough guide on T4 free: 1 hour ≈ 3 epochs at 1,000 pairs.

## Tracking total epochs

Cellpose doesn't store epoch count in the checkpoint. Keep your own
rough tally in the markdown below or in a sidecar text file on Drive
(`models/cpsam_dic_epochs.txt`).

After two sessions: original 6 + this run 14 = ~20 total.